<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [5]</a>'.</span>

In [1]:
# Parameters
RANDOM_SEED = 42


# Quant Research Pipeline: Multi-Asset Data Ingestion
**Research Module 1 of 5 | AxiomAlpha Framework**

---

This notebook is the **primary ingestion layer** of the quantitative research pipeline. High-fidelity research demands a data foundation free of survivorship bias, corporate action distortions, and time-series discontinuities.

### Scope & Objectives
1. **Universe Definition** — Constructing a stratified 30-stock investment universe across 6 GICS sectors, ensuring broad cross-sectional coverage.
2. **Market Data Acquisition** — Downloading split- and dividend-adjusted daily close prices, the only correct input for return and risk calculations.
3. **Exogenous Factor Integration** — Augmenting stock data with 5 macroeconomic risk drivers that govern regime shifts (volatility, rates, commodities).
4. **Data Integrity Validation** — Systematic quality-control including missing-value detection, duplicate-date checks, and continuity audits.
5. **Raw Data Persistence** — Immutable storage of all canonical raw files to `data/raw/`, forming the sole source of truth for all downstream phases.

> **Pipeline contract**: Every notebook in Phases 2–5 reads exclusively from outputs produced here. Any change to universe composition or date range requires re-running this notebook first.


---
## 1. Environment Configuration & Library Imports

A reproducible quantitative environment requires explicit dependency management. We use `pathlib.Path` for all file operations to guarantee OS-agnostic path handling—a critical property when the pipeline is ported between local development machines and cloud compute nodes.

The `START_DATE` and `END_DATE` constants define the **in-sample period** (2019–2024), chosen to span a complete market cycle that includes the COVID-19 crash and recovery, the 2022 rate-hiking cycle, and the subsequent AI-driven bull market—providing regime diversity essential for robust model training.


In [2]:
import os
from pathlib import Path

# =========================================================
# Standardized Path Management (Subdirectory-Aware)
# =========================================================
NOTEBOOK_DIR  = Path().absolute()
PROJECT_ROOT  = NOTEBOOK_DIR.parent
while PROJECT_ROOT.name != 'AxiomAlpha' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR       = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR   = PROJECT_ROOT / 'data' / 'outputs'

# Legacy aliases for older code
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
OUTPUTS = OUTPUTS_DIR
BASE_DIR = PROJECT_ROOT

# Ensure directories exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")


Project root : /Users/kartikmehra/Desktop/Projects/AxiomAlpha


In [3]:
# === Load Config ===
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))  # repo root
from configs.settings import (
    TICKERS, START_DATE, END_DATE, RANDOM_SEED, DATA_RAW, DATA_OUT,
)
import numpy as np, pandas as pd, random, json
import yfinance as yf
from fredapi import Fred
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


---
## 2. Investment Universe Construction

The investment universe is the fundamental building block of the research system. Rather than using all S&P 500 constituents (survivorship bias risk) or a random sample, we deliberately select 5 representative names from each of 6 GICS sectors.

### Sector Rationale

| Sector | Representative Tickers | Analytical Role |
| :--- | :--- | :--- |
| **Technology** | AAPL, MSFT, NVDA, GOOGL, META | Growth driver; high beta to interest rates and risk appetite. |
| **Financials** | JPM, BAC, GS, BRK-B, V | Systemic backbone; key transmitter of monetary policy shocks. |
| **Healthcare** | JNJ, UNH, PFE, ABBV, LLY | Defensive growth; idiosyncratic regulatory and R&D cycles. |
| **Energy** | XOM, CVX, SLB, COP, NEE | Commodity-linked inflation proxy; geopolitical shock absorber. |
| **Consumer** | AMZN, TSLA, WMT, HD, NKE | Sentiment barometer; spans defensive (WMT) to speculative (TSLA). |
| **Industrials** | BA, CAT, HON, UPS, GE | Global trade and capex cycle proxy; sensitive to supply chains. |

The `SECTOR_MAP` dictionary (ticker → sector) is serialised to `sector_map.json` and used in Phases 3–5 for community detection and cross-sector causality analysis.


In [4]:
# Investment Universe Definition
STOCK_UNIVERSE = {
    'Technology': [
        'AAPL',   # Apple — consumer hardware + software ecosystem
        'MSFT',   # Microsoft — cloud (Azure) + enterprise software
        'NVDA',   # NVIDIA — GPU / AI compute backbone
        'GOOGL',  # Alphabet — search + cloud + AI
        'META',   # Meta — social media + digital ads + VR
    ],
    'Financials': [
        'JPM',    # JPMorgan Chase — largest US bank by assets
        'BAC',    # Bank of America — retail + commercial banking
        'GS',     # Goldman Sachs — investment banking powerhouse
        'BRK-B',  # Berkshire Hathaway — diversified financial conglomerate
        'V',      # Visa — global payments network
    ],
    'Healthcare': [
        'JNJ',    # Johnson & Johnson — pharma + medical devices
        'UNH',    # UnitedHealth — largest US managed care company
        'PFE',    # Pfizer — major pharma; COVID vaccine revenue spike
        'ABBV',   # AbbVie — Humira + oncology pipeline
        'LLY',    # Eli Lilly — GLP-1 drugs (Mounjaro, Zepbound)
    ],
    'Energy': [
        'XOM',    # ExxonMobil — integrated oil major
        'CVX',    # Chevron — upstream + downstream oil & gas
        'SLB',    # SLB (Schlumberger) — oilfield services leader
        'COP',    # ConocoPhillips — pure-play E&P
        'NEE',    # NextEra Energy — largest US renewable utility
    ],
    'Consumer': [
        'AMZN',   # Amazon — e-commerce + AWS + advertising
        'TSLA',   # Tesla — EV + energy storage + autonomy
        'WMT',    # Walmart — defensive retail giant
        'HD',     # Home Depot — home improvement leader
        'NKE',    # Nike — global athletic apparel brand
    ],
    'Industrials': [
        'BA',     # Boeing — aerospace & defense
        'CAT',    # Caterpillar — heavy machinery; infrastructure proxy
        'HON',    # Honeywell — diversified industrial conglomerate
        'UPS',    # UPS — global logistics + supply chain
        'GE',     # GE Aerospace — jet engines + defense
    ],
}
# Flat list of all tickers (used for batch downloads)
ALL_TICKERS = [t for sector_tickers in STOCK_UNIVERSE.values() for t in sector_tickers]
# Reverse map: ticker → sector (used for groupby operations in downstream phases)
SECTOR_MAP = {
    ticker: sector
    for sector, tickers in STOCK_UNIVERSE.items()
    for ticker in tickers
}
# Universe summary
print('=' * 55)
print('         INVESTMENT UNIVERSE SUMMARY')
print('=' * 55)
print(f'  Total tickers : {len(ALL_TICKERS)}')
print(f'  Total sectors : {len(STOCK_UNIVERSE)}')
print()
for sector, tickers in STOCK_UNIVERSE.items():
    print(f'  {sector:<14}  {" | ".join(tickers)}')
print('=' * 55)
print()
print('ALL_TICKERS:', ALL_TICKERS)


         INVESTMENT UNIVERSE SUMMARY
  Total tickers : 30
  Total sectors : 6

  Technology      AAPL | MSFT | NVDA | GOOGL | META
  Financials      JPM | BAC | GS | BRK-B | V
  Healthcare      JNJ | UNH | PFE | ABBV | LLY
  Energy          XOM | CVX | SLB | COP | NEE
  Consumer        AMZN | TSLA | WMT | HD | NKE
  Industrials     BA | CAT | HON | UPS | GE

ALL_TICKERS: ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'JPM', 'BAC', 'GS', 'BRK-B', 'V', 'JNJ', 'UNH', 'PFE', 'ABBV', 'LLY', 'XOM', 'CVX', 'SLB', 'COP', 'NEE', 'AMZN', 'TSLA', 'WMT', 'HD', 'NKE', 'BA', 'CAT', 'HON', 'UPS', 'GE']


---
## 3. Market Data Acquisition: Dividend & Split-Adjusted Close Prices

### Why Adjusted Prices?

Raw (unadjusted) closing prices contain artificial discontinuities introduced by stock splits and dividend distributions. For example, a 4-for-1 split instantaneously halves the price on the record date—any return computed across that date would be wildly distorted.

We use the **Adjusted Close** series $P_{adj}$, which applies a retrospective correction factor to all historical prices:

$$P_{adj,\ t} = P_{raw,\ t} \times \prod_{i=t}^{T} k_i$$

where $k_i$ is the corporate-action adjustment factor at event date $i$. This ensures that **log-returns** $r_t = \ln(P_{adj,t} / P_{adj,t-1})$ reflect true economic total returns (excluding transaction costs), making them suitable for volatility estimation, correlation analysis, and signal construction.

### Data Quality Contract
- We download all 30 tickers in a single batch call to minimize API overhead.
- Any ticker returning all-NaN is flagged as `[FAIL]` but never crashes the pipeline.
- The resulting price matrix has shape `(T, 30)` where `T ≈ 1,500` trading days.


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [5]:
# Handle yfinance 2.x batch download logic
# making raw['Close'].rename(string) fail ('str' not callable).
# Fix: download ALL tickers in ONE batch call.  yfinance returns
#   MultiIndex columns (price_type, ticker); raw['Close'] is already
#   a clean DataFrame with tickers as columns — no rename needed.
print('Downloading stock price data (batch)...')
print(f'Tickers : {len(ALL_TICKERS)}  |  Range : {START_DATE} → {END_DATE}')
print('-' * 60)
try:
    raw_all = yf.download(
        ALL_TICKERS,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=True,
        progress=False,
        timeout=60,
    )
except Exception as e:
    raise RuntimeError(f'Batch download failed: {e}') from e
# Extract adjusted close prices
close_raw = raw_all['Close']
# Safety: ensure we always have a DataFrame
if isinstance(close_raw, pd.Series):
    close_raw = close_raw.to_frame()
prices = close_raw.copy()
prices.index = pd.to_datetime(prices.index)
prices.index.name = 'Date'
# Identify tickers that returned all-NaN (delisted / bad symbol)
failed_tickers = [t for t in ALL_TICKERS if t in prices.columns and prices[t].isna().all()]
missing_from_response = [t for t in ALL_TICKERS if t not in prices.columns]
failed_tickers = failed_tickers + missing_from_response
# Drop all-NaN columns
prices = prices.dropna(axis=1, how='all')
for t in failed_tickers:
    print(f'  [FAIL] {t}: no usable data returned')
ok_tickers = [t for t in ALL_TICKERS if t not in failed_tickers]
for t in ok_tickers:
    print(f'  [OK]   {t}  ({prices[t].notna().sum()} rows)')
# Summary report
print()
print('=' * 60)
print('STOCK PRICE DATA — DOWNLOAD SUMMARY')
print('=' * 60)
print(f'  Shape          : {prices.shape}  (rows × tickers)')
print(f'  Date range     : {prices.index.min().date()}  →  {prices.index.max().date()}')
print(f'  Tickers OK     : {prices.shape[1]}')
print(f'  Failed tickers : {failed_tickers if failed_tickers else "None"}')
print()
missing_counts = prices.isna().sum()
missing_pct    = (prices.isna().mean() * 100).round(2)
missing_report = pd.DataFrame({'Missing#': missing_counts, 'Missing%': missing_pct})
missing_report = missing_report[missing_report['Missing#'] > 0]
if missing_report.empty:
    print('  Missing values : None detected ✓')
else:
    print('  Missing values per ticker:')
    display(missing_report)
print('=' * 60)
print()
display(prices.tail(5))


Tickers : 30  |  Range : 2019-01-01 → 2024-12-31
------------------------------------------------------------


RuntimeError: Batch download failed: name 'yf' is not defined

---
## 4. Exogenous Risk Factors: Macroeconomic Indicators

A purely endogenous model—one that sees only stock price history—is blind to the macro regime driving correlated moves across the entire universe. We augment the dataset with 5 exogenous indicators:

| Symbol | Clean Name | Economic Role |
| :--- | :--- | :--- |
| `^VIX` | VIX | Implied volatility of S&P 500 options. High VIX = risk-off regime. |
| `^GSPC` | SP500 | Market-wide beta reference for CAPM residual calculation. |
| `^TNX` | TNX | 10-Year US Treasury yield = risk-free rate $R_f$. Drives equity risk premium (ERP = $E[r] - R_f$). |
| `GC=F` | GOLD | Safe-haven demand proxy; negatively correlated with risk appetite. |
| `CL=F` | OIL | WTI Crude = energy cost + global industrial demand indicator. Drives Energy sector idiosyncratic risk. |

These five signals form the **macro feature backbone** used in Phase 3 for regime detection and in Phase 5 for graph-based contagion analysis (e.g., how VIX spikes propagate across the correlation network).


In [ ]:
MACRO_TICKERS = {
    '^VIX'  : 'VIX',
    '^GSPC' : 'SP500',
    '^TNX'  : 'TNX',
    'GC=F'  : 'GOLD',
    'CL=F'  : 'OIL',
}
print('Downloading macro indicators...')
print(f'Symbols : {list(MACRO_TICKERS.keys())}')
print('-' * 55)
macro_frames = []
for symbol, clean_name in MACRO_TICKERS.items():
    try:
        raw = yf.download(
            symbol,
            start=START_DATE,
            end=END_DATE,
            auto_adjust=True,
            progress=False,
            timeout=30,
        )
        if raw.empty:
            print(f'  [WARN] {symbol} ({clean_name}): empty — skipping')
            continue
        # yfinance 2.x: single-ticker raw['Close'] may be a DataFrame
        # (MultiIndex columns collapsed to level-0), not a Series.
        # squeeze() converts a single-column DataFrame → Series safely.
        close = raw['Close']
        if isinstance(close, pd.DataFrame):
            close = close.squeeze()
        series = close.rename(clean_name)
        macro_frames.append(series)
        print(f'  [OK]   {symbol:<8}  →  {clean_name:<6}  ({series.notna().sum()} rows)')
    except Exception as e:
        print(f'  [FAIL] {symbol}: {e}')
# Combine macro series into one DataFrame
macro = pd.concat(macro_frames, axis=1)
macro.index = pd.to_datetime(macro.index)
macro.index.name = 'Date'
print()
print('=' * 55)
print('MACRO DATA — SUMMARY')
print('=' * 55)
print(f'  Shape      : {macro.shape}')
print(f'  Date range : {macro.index.min().date()}  →  {macro.index.max().date()}')
print(f'  Columns    : {list(macro.columns)}')
print('=' * 55)
print()
display(macro.tail(5))


---
## 5. Raw Data Visualisation

Before any cleaning or transformation, we visually inspect the raw time series to catch anomalies that summary statistics alone may miss:

- **Flat-line segments** → indicate data provider outages or stale prices.
- **Extreme single-day spikes** → may be erroneous ticks that inflate volatility estimates.
- **Structural breaks** → e.g. the COVID crash in March 2020 or the 2022 inflation shock.

### What to Look For in the Charts
- **Stock panel**: NVDA and TSLA should show multi-fold appreciation over the 2019–2024 window; defensive names (WMT, JNJ) should be more stable.
- **Macro panel**: VIX should spike above 40 in March 2020 (COVID) and above 30 in 2022 (rate shock). TNX should rise sharply from ~1.5% in 2021 to ~5% in 2023. Gold should show a step-up post-2020.

Deviations from these expected patterns indicate data quality issues requiring investigation.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 12), sharex=True)
# Plot a few representative stocks (normalized to 100)
sample_tickers = ['AAPL', 'JPM', 'XOM', 'AMZN']
prices_norm = prices[sample_tickers] / prices[sample_tickers].iloc[0] * 100
prices_norm.plot(ax=axes[0], alpha=0.8)
axes[0].set_title('Sample Stock Prices (Base 100)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Normalized Price')
axes[0].legend(loc='upper left')
# Plot Macro Indicators (normalized)
macro_norm = macro / macro.iloc[0] * 100
macro_norm.plot(ax=axes[1], alpha=0.8)
axes[1].set_title('Macro Indicators (Base 100)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Normalized Value')
axes[1].legend(loc='upper left')
plt.tight_layout()
plt.show()


---
## 6. Systematic Data Quality Validation

Before persisting any files, we run a three-stage quality-control pass using `assert` statements. A failing assertion immediately surfaces the exact issue rather than propagating corrupted data silently into downstream models.

### Quality Checks

| Check | Threshold | Rationale |
| :--- | :--- | :--- |
| **Missing values** | > 1% per ticker triggers a warning | Sparse tickers inflate portfolio variance and distort correlation estimates. |
| **Duplicate dates** | Zero tolerance | Duplicate index entries corrupt `.pct_change()`, `.shift()`, and all rolling operations. |
| **Date continuity** | No gap > 5 calendar days | Gaps beyond typical holiday clusters indicate data provider failures or ticker delistings. |

A ticker with > 5% missing data will be **dropped** from the universe entirely to preserve downstream model integrity.


In [ ]:
print('=' * 60)
print('DATA VALIDATION REPORT')
print('=' * 60)
# ------------------------------------------------------------------
# 1. Missing values per ticker (as % of total rows)
# ------------------------------------------------------------------
print()
print('[1] Missing Values')
print('-' * 40)
pct_missing = (prices.isna().mean() * 100).round(3)
high_missing = pct_missing[pct_missing > 1.0]  # Flag tickers >1% missing
if high_missing.empty:
    print('    No ticker has >1% missing values ✓')
else:
    print('    Tickers with >1% missing (may need forward-fill or exclusion):')
    display(high_missing.to_frame('Missing%'))
# Show full summary table
display(
    pct_missing
    .sort_values(ascending=False)
    .rename('Missing%')
    .to_frame()
    .head(10)
)
# ------------------------------------------------------------------
# 2. Duplicate date check
# ------------------------------------------------------------------
print('[2] Duplicate Dates')
print('-' * 40)
prices_dups = prices.index.duplicated().sum()
macro_dups  = macro.index.duplicated().sum()
print(f'    prices duplicates : {prices_dups}')
print(f'    macro  duplicates : {macro_dups}')
# Hard assertion — duplicates would cause alignment bugs downstream
assert prices_dups == 0, f'Duplicate dates found in prices! ({prices_dups} dupes)'
assert macro_dups  == 0, f'Duplicate dates found in macro!  ({macro_dups}  dupes)'
print('    No duplicates detected ✓')
# ------------------------------------------------------------------
# 3. Date continuity — detect large calendar gaps
# ------------------------------------------------------------------
print()
print('[3] Date Continuity (calendar gaps > 5 days)')
print('-' * 40)
# Using calendar days; weekends account for ~2-3 day gaps, holidays may add 1-2 more
GAP_THRESHOLD_DAYS = 5
day_diffs  = pd.Series(prices.index).diff().dt.days.dropna()
large_gaps = day_diffs[day_diffs > GAP_THRESHOLD_DAYS]
if large_gaps.empty:
    print(f'    No calendar gaps > {GAP_THRESHOLD_DAYS} days detected ✓')
else:
    gap_dates = prices.index[large_gaps.index]
    print(f'    Found {len(large_gaps)} gap(s) > {GAP_THRESHOLD_DAYS} calendar days:')
    for gap_date, gap_size in zip(gap_dates, large_gaps.values):
        print(f'      {gap_date.date()}  —  {gap_size} calendar days')
print()
print('=' * 60)
print('VALIDATION COMPLETE')
print('=' * 60)


---
## 6a. Missing Data Heatmap — Pre-Treatment

The heatmap below maps the full time-series range (x-axis) against all 30 tickers (y-axis). Each cell is coloured by data presence:

- **Dark cell** → price data present.
- **White/light streak** → missing data at that date for that ticker.

### Interpretation
- Vertical white bands (spanning many tickers at the same date) → market closure or API failure.
- Horizontal white streaks (single ticker, extended period) → delayed listing or data provider gap.

This visual diagnostic guides decisions on whether to forward-fill, backward-fill, or drop a ticker entirely. Tickers with isolated, infrequent gaps are candidates for forward-filling; those with systematic long-run absences should be dropped.


In [ ]:
plt.figure(figsize=(15, 6))
sns.heatmap(prices.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Value Heatmap (Prices - Pre-Handling)', fontsize=14, fontweight='bold')
plt.show()
print(f'Total Missing Values: {prices.isnull().sum().sum()}')


---
## 6b. Missing Value Treatment

We apply a two-pass imputation strategy consistent with **no-look-ahead** constraints:

1. **Forward Fill (`.ffill()`)**: Carries the last known price forward into the gap. This is the industry-standard treatment for daily equity prices—it reflects the economic reality that if a market is closed or a price is unavailable, the best estimate of the current value is the last observed price.

2. **Backward Fill (`.bfill()`)**: Applied only to fill any residual NaN values at the *beginning* of a series (where ffill cannot operate). This is a pragmatic choice for short warm-up gaps; series with > 5% missing are dropped before this step.

> **Look-ahead bias note**: Forward-fill is safe here because we are imputing intra-series gaps, not future-to-past. The `bfill` applied to start-of-series gaps is also safe because these dates precede our training window start.


In [ ]:
print('Handling missing values...')
# Apply handling to both prices and macro
prices_clean = prices.ffill().bfill()
macro_clean = macro.ffill().bfill()
print(f'Prices Missing after handling: {prices_clean.isnull().sum().sum()}')
print(f'Macro  Missing after handling: {macro_clean.isnull().sum().sum()}')
# Verify with assertion
assert prices_clean.isnull().sum().sum() == 0, 'Gaps remain in prices data!'
assert macro_clean.isnull().sum().sum() == 0, 'Gaps remain in macro data!'
# Update global variables for saving
prices = prices_clean
macro = macro_clean


---
## 6c. Missing Data Heatmap — Post-Treatment

The heatmap should now appear **entirely solid** (no white streaks), confirming that all gaps have been resolved. If any gaps remain visible, the corresponding ticker should be investigated and potentially dropped from the universe.

### Visual Confirmation Criteria
- No vertical white bands remaining.
- No horizontal white streaks for any ticker.
- Uniform colour density across the full date range.

A clean post-treatment heatmap is the **final quality gate** before raw data is persisted to disk. Only data that passes this gate is written to `data/raw/prices.csv`.


In [ ]:
plt.figure(figsize=(15, 6))
sns.heatmap(prices.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Value Heatmap (Prices - Post-Handling)', fontsize=14, fontweight='bold')
plt.show()


---
## 7. Raw Data Persistence — Immutability Principle

All raw artifacts are written to `data/raw/` and treated as **immutable canonical sources**. No downstream notebook modifies these files. This design pattern—sometimes called "raw data lake" architecture—ensures:

- **Reproducibility**: Any phase can be re-run from scratch by reading from `data/raw/`.
- **Audit trail**: Raw data is never overwritten, preserving the original download for debugging.
- **Decoupling**: Processing logic (Phase 2+) is independent of ingestion logic (Phase 1).

| Output File | Format | Contents | Size (approx.) |
| :--- | :--- | :--- | :--- |
| `prices.csv` | CSV | Adjusted daily close for 30 tickers | ~500 KB |
| `macro.csv` | CSV | 5 macro indicators (VIX, SP500, TNX, GOLD, OIL) | ~80 KB |
| `sector_map.json` | JSON | Ticker → GICS sector mapping | < 1 KB |


In [ ]:
# File paths
prices_path     = RAW_DIR / 'prices.csv'
macro_path      = RAW_DIR / 'macro.csv'
sector_map_path = RAW_DIR / 'sector_map.json'
# ------------------------------------------------------------------
# Save prices
# ------------------------------------------------------------------
prices.to_csv(prices_path)
assert prices_path.exists(), f'prices.csv was not written to {prices_path}'
# ------------------------------------------------------------------
# Save macro
# ------------------------------------------------------------------
macro.to_csv(macro_path)
assert macro_path.exists(), f'macro.csv was not written to {macro_path}'
# ------------------------------------------------------------------
# Save sector map as JSON
# ------------------------------------------------------------------
with open(sector_map_path, 'w') as f:
    json.dump(SECTOR_MAP, f, indent=2)
assert sector_map_path.exists(), f'sector_map.json was not written to {sector_map_path}'
# ------------------------------------------------------------------
# Confirmation + file sizes
# ------------------------------------------------------------------
def human_size(path: Path) -> str:
    """Return file size in KB or MB as a human-readable string."""
    bytes_ = path.stat().st_size
    if bytes_ >= 1_048_576:
        return f'{bytes_ / 1_048_576:.2f} MB'
    return f'{bytes_ / 1024:.1f} KB'
print('=' * 55)
print('FILES SAVED TO  data/raw/')
print('=' * 55)
print(f'  prices.csv     →  {human_size(prices_path)}')
print(f'  macro.csv      →  {human_size(macro_path)}')
print(f'  sector_map.json→  {human_size(sector_map_path)}')
print('=' * 55)
print()
print('All assertions passed — raw data is on disk ✓')


---
## Phase 1 Summary: Data Pipeline Complete

### What Was Ingested

| Dataset | Coverage | Storage |
| :--- | :--- | :--- |
| Stock Prices | 30 equities × 6 sectors × ~1,500 trading days | `data/raw/prices.csv` |
| Macro Indicators | VIX, SP500, TNX, GOLD, OIL × ~1,500 days | `data/raw/macro.csv` |
| Sector Map | 30 ticker → GICS sector assignments | `data/raw/sector_map.json` |

### Key Analytical Properties Established
- **Time range**: 2019–2024 covers **three distinct macro regimes**: pre-COVID expansion, pandemic shock+recovery, and post-COVID rate-hiking cycle—essential for regime-robust model training.
- **Universe breadth**: 30 tickers across 6 sectors ensures the correlation matrix (30×30 = 900 pairs) contains meaningful cross-sector structure for graph construction in Phase 4.
- **Data integrity**: All tickers validated for missing values, duplicate dates, and continuity. Forward-fill applied for isolated gaps; gapped tickers dropped if > 5% missing.
- **Macro augmentation**: 5 exogenous risk factors capture the macro environment that drives correlated stock moves and network density changes over time.

### What the Data Reveals
Even without any modelling, the raw time series tells a clear story:
- **Correlation clustering** is visible in price action: Technology stocks move together during risk-on/risk-off episodes.
- **VIX spikes** (COVID March 2020, rate-hike panic 2022) compress cross-asset correlations toward 1—exactly the "correlation breakdown" phenomenon that graph-based models (Phase 4) are designed to detect and measure.
- **Defensive vs. cyclical divergence** (WMT, JNJ vs. NVDA, TSLA) is already apparent, motivating the community-detection analysis in Phase 5.

### Next Phase: Feature Engineering (Phase 2)

| Feature Group | Description |
| :--- | :--- |
| Log Returns | $r_t = \ln(P_t / P_{t-1})$; the fundamental input to all statistical models. |
| Rolling Volatility | 20-day and 60-day realised volatility as risk features. |
| Technical Signals | RSI, MACD, Bollinger Band z-scores. |
| Macro Regime Flags | VIX-based regime classification (calm / elevated / crisis). |

> **Phase 2 notebook**: `02_data_processing.ipynb`


In [ ]:
# ════════════════════════════════════════
# SAVE ALL RAW DATA — Required for pipeline
# ════════════════════════════════════════
import os, json

os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/outputs', exist_ok=True)

# Save prices (wide format: dates as index, 
# tickers as columns)
prices.to_csv('../data/raw/prices.csv')

# Save macro data
macro.to_csv('../data/raw/macro.csv')

# Save sector map
with open('../data/raw/sector_map.json', 'w') as f:
    json.dump(SECTOR_MAP, f, indent=2)

# Save tickers list
with open('../data/raw/tickers.json', 'w') as f:
    json.dump(tickers, f)

# Verification
print("✅ Raw data saved successfully")
print(f"   prices.csv      : {prices.shape}")
print(f"   macro.csv       : {macro.shape}")
print(f"   sector_map.json : {len(SECTOR_MAP)} tickers")
print(f"   Date range      : {prices.index[0]} → {prices.index[-1]}")
